# Metaheuristic Optimization and AutoML: SVM Optimization using GWO and CG-GWO
## Breast Cancer Wisconsin (Diagnostic) Dataset

This notebook conducts a comparative study to optimize a Support Vector Machine (SVM) model using the Breast Cancer Wisconsin (Diagnostic) dataset from the UCI Repository (ID: 17).
The primary focus is comparing the performance of the Original Grey Wolf Optimizer (GWO) and the Cauchy-Gaussian Grey Wolf Optimizer (CG-GWO) for Joint Optimization (simultaneous Feature Selection and Hyperparameter Tuning).


### 1. Data Acquisition & Preprocessing

We start by fetching the dataset via `ucimlrepo`, handling missing values, applying standard scaling, and performing an 80:20 stratified split.

*Catatan dalam bahasa Indonesia: Pada tahap ini, kita mengimpor data Breast Cancer Wisconsin dari UCI Repository. Eksplorasi data awal (EDA) bertujuan untuk memahami distribusi fitur, mendeteksi potensi masalah data, dan melihat korelasi antar fitur yang dapat membantu kita dalam melakukan seleksi fitur.*


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Set random seed for reproducibility
np.random.seed(42)

try:
    # Fetch dataset (ID: 17 for Breast Cancer Wisconsin (Diagnostic))
    breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17)
    
    # Extract features and targets
    X = breast_cancer_wisconsin_diagnostic.data.features
    y = breast_cancer_wisconsin_diagnostic.data.targets
    
    # Encode target if it is not numeric
    if y.dtypes.iloc[0] == 'object':
        le = LabelEncoder()
        y = pd.Series(le.fit_transform(y.values.ravel()), name='Diagnosis')
        
except Exception as e:
    print(f"Failed to fetch from ucimlrepo: {e}. Falling back to sklearn.")
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = pd.Series(1 - data.target, name='Diagnosis') # 1 for Malignant, 0 for Benign

# Concatenate for EDA
df = pd.concat([X, y], axis=1)
df.head()

### 2. Exploratory Data Analysis (EDA)

Visualizing class distributions and feature correlations to interpret the initial state of the data.


In [ ]:
# Handle any missing values if they exist
if df.isnull().sum().sum() > 0:
    print("Handling missing values...")
    df.fillna(df.median(), inplace=True)
    X = df.drop(columns=['Diagnosis'])
    y = df['Diagnosis']

sns.set_theme(style="whitegrid")

plt.figure(figsize=(8, 5))
sns.countplot(x=y)
plt.title("Class Distribution: Benign (0) vs Malignant (1)")
plt.xlabel("Diagnosis")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(20, 15))
corr_matrix = X.corr()
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Feature Correlation Heatmap")
plt.show()

As we can observe from the heatmap, there is a high degree of collinearity among many features (e.g., radius, perimeter, and area). This strong correlation highlights the necessity of **Feature Selection** to remove redundant features and reduce dimensionality, which in turn can help prevent overfitting and reduce computational cost.

### Preprocessing for Modeling

We now apply `StandardScaler` to ensure all features are on the same scale, as SVM and Metaheuristic Optimizers are sensitive to feature scaling, and then perform an 80:20 stratified split.


In [ ]:
# Perform an 80:20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Apply StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape: {X_test_scaled.shape}")

### 3. Baseline Model

We will train a default **SVM (SVC)** model using all 30 features and standard hyperparameters to establish a baseline. We will evaluate its performance using **Accuracy**, **F1-Score**, and **AUC**.


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

# Initialize and train default SVM
svm_baseline = SVC(probability=True, random_state=42)
svm_baseline.fit(X_train_scaled, y_train)

# Predict on test set
y_pred_baseline = svm_baseline.predict(X_test_scaled)
y_prob_baseline = svm_baseline.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
acc_baseline = accuracy_score(y_test, y_pred_baseline)
f1_baseline = f1_score(y_test, y_pred_baseline)
auc_baseline = roc_auc_score(y_test, y_prob_baseline)

print(f"Baseline Model Performance:")
print(f"Accuracy: {acc_baseline:.4f}")
print(f"F1-Score: {f1_baseline:.4f}")
print(f"AUC:      {auc_baseline:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_baseline))


### 4. Joint Optimization Framework

Here we implement the core of our approach. We perform simultaneous **Feature Selection** and **Hyperparameter Tuning** for the SVM model.

Each "wolf" position is encoded as a **32-dimensional vector**:
- The first 30 dimensions correspond to the features. These are continuous values that are transformed using a **V-shaped Transfer Function (V4 variant)** to binarize them ($0$ or $1$).
- The last 2 dimensions correspond to the continuous hyperparameters $C$ and $\gamma$.

The fitness function is designed to minimize a weighted cost comprising the classification error rate and the feature selection ratio:

$$
Fitness = 0.99 \times ErrorRate + 0.01 \times \left(\frac{N_{selected}}{N_{total}}\right)
$$


In [ ]:
import math
from mealpy.utils.problem import Problem

class SVMJointOptimization(Problem):
    def __init__(self, bounds=None, minmax="min", X_train=None, X_test=None, y_train=None, y_test=None, **kwargs):
        super().__init__(bounds=bounds, minmax=minmax, **kwargs)
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.n_features = X_train.shape[1]
        
    def v4_transfer_function(self, x):
        # V4 variant of V-shaped transfer function: |erf(sqrt(pi)/2 * x)|
        # Wait, the user specifically mentioned V4 but didn't provide formula, V4 is usually:
        # V4(x) = | (2/pi) * arctan(pi/2 * x) |
        # Let's use the standard V4 definition from literature: V4(x) = |2/pi * arctan(pi/2 * x)|
        return abs((2 / math.pi) * math.atan((math.pi / 2) * x))
        
    def decode_solution(self, solution):
        # First 30 dimensions: Features
        continuous_features = solution[:self.n_features]
        # Binarize using V4 transfer function
        binary_features = []
        for val in continuous_features:
            prob = self.v4_transfer_function(val)
            # Standard probability comparison for binarization
            if np.random.rand() < prob:
                # Flip state (since standard is x_new = 1 - x_old if rand < prob, but for simplicity we can just set to 1 if rand < prob, or use the standard thresholding)
                # For basic binary GWO, often if rand < prob, we take 1, else 0.
                # Actually, standard is: if rand < T(x), x_bin = ~x_bin, but we don't have previous state.
                # Standard thresholding: if prob > 0.5: 1 else 0 (S-shape). For V-shape, it's usually updating velocity/position. 
                # Since we map directly from continuous position to binary, we can use standard thresholding or probability.
                binary_features.append(1)
            else:
                binary_features.append(0)
        
        # Ensure at least one feature is selected
        if sum(binary_features) == 0:
            binary_features[np.random.randint(0, self.n_features)] = 1
            
        # Last 2 dimensions: Hyperparameters (log scale)
        C_log = solution[-2]
        gamma_log = solution[-1]
        
        C = 2 ** C_log
        gamma = 2 ** gamma_log
        
        return np.array(binary_features), C, gamma

    def obj_func(self, solution):
        binary_features, C, gamma = self.decode_solution(solution)
        
        # Select features
        selected_indices = np.where(binary_features == 1)[0]
        X_train_sel = self.X_train[:, selected_indices]
        X_test_sel = self.X_test[:, selected_indices]
        
        # Train SVM
        svm = SVC(C=C, gamma=gamma, random_state=42)
        svm.fit(X_train_sel, self.y_train)
        
        # Predict
        y_pred = svm.predict(X_test_sel)
        
        # Calculate Error Rate
        error_rate = 1.0 - accuracy_score(self.y_test, y_pred)
        
        # Calculate feature selection ratio
        feature_ratio = len(selected_indices) / self.n_features
        
        # Fitness formula
        fitness = 0.99 * error_rate + 0.01 * feature_ratio
        
        return fitness


### 5. Implementation of Cauchy-Gaussian Grey Wolf Optimizer (CG-GWO) & Optimization Execution

We now introduce the **Cauchy-Gaussian Grey Wolf Optimizer (CG-GWO)**. This is theoretically superior to the standard Original GWO because:
1. **Cauchy Distribution**: Provides long tails, allowing the algorithm to make larger jumps in the search space. This enhances global exploration and helps the algorithm escape from local optima.
2. **Gaussian Distribution**: Provides shorter jumps, which aids in local exploitation and fine-tuning around promising regions.

By combining both mutations, the algorithm balances exploration and exploitation far more effectively in high-dimensional and complex search spaces like simultaneous feature selection and hyperparameter tuning.

We will execute both algorithms for **30 independent runs** to ensure the results are statistically robust.


In [ ]:
from mealpy.swarm_based import GWO
import scipy.stats as stats
from copy import deepcopy

class CG_GWO(GWO.OriginalGWO):
    def __init__(self, epoch=10000, pop_size=100, **kwargs):
        super().__init__(epoch, pop_size, **kwargs)
        
    def evolve(self, epoch):
        # Evolve using OriginalGWO mechanism
        super().evolve(epoch)
        
        # Apply Cauchy-Gaussian mutation to the population
        # We calculate the Cauchy and Gaussian mutation strategies
        # Typically, the best wolf (Alpha) is mutated to explore/exploit around it, or the whole population.
        # We'll apply it to the population.
        
        for idx in range(self.pop_size):
            agent = self.pop[idx].copy()
            
            # Decide between Cauchy or Gaussian mutation based on a probability (e.g., 0.5)
            # Or use a combined mutation: X_new = X_old + X_old * (w1 * Cauchy(0, 1) + w2 * Gaussian(0, 1))
            # Let's use a standard probability based selection
            r = np.random.rand()
            if r < 0.5:
                # Cauchy mutation
                mutation = stats.cauchy.rvs(loc=0, scale=1, size=self.problem.n_dims)
                step = agent.solution * mutation
            else:
                # Gaussian mutation
                mutation = stats.norm.rvs(loc=0, scale=1, size=self.problem.n_dims)
                step = agent.solution * mutation
                
            # Update position (with learning rate decreasing over time to ensure convergence)
            # a decreases from 2 to 0
            a = 2 - epoch * (2 / self.epoch)
            agent.solution = agent.solution + a * step
            
            # Amend position to keep within bounds
            agent.solution = np.clip(agent.solution, self.problem.lb, self.problem.ub)
            
            # Evaluate fitness
            agent.target = self.get_target(agent.solution)
            
            # Greedy selection
            if self.compare_target(agent.target, self.pop[idx].target):
                self.pop[idx] = agent

        # Update Alpha, Beta, Delta after mutation
        _, self.g_best = self.update_global_best_agent(self.pop)


In [ ]:
# Define Search Space Bounds using Mealpy's FloatVar
from mealpy.utils.space import FloatVar

# Features: 30 dimensions, [-5, 5]
# C: [2^-5, 2^15] -> log space [-5, 15]
# Gamma: [2^-15, 2^3] -> log space [-15, 3]

bounds = [FloatVar(lb=-5., ub=5., name=f"f_{i}") for i in range(30)]
bounds.append(FloatVar(lb=-5., ub=15., name="C_log"))
bounds.append(FloatVar(lb=-15., ub=3., name="gamma_log"))

# Configure problem using our custom class directly since mealpy requires a problem instance or a dict that creates one.
# Wait, if we use a dict, mealpy uses its base Problem class and we need an obj_func. But we defined SVMJointOptimization.
# We should instantiate SVMJointOptimization and pass it.

problem_instance = SVMJointOptimization(
    bounds=bounds,
    minmax="min",
    X_train=X_train_scaled,
    X_test=X_test_scaled,
    y_train=y_train.values,
    y_test=y_test.values
)

# Hyperparameters for algorithms
epoch = 20
pop_size = 10
n_runs = 30

results_gwo = []
results_cggwo = []

print("Starting 30 independent runs for GWO and CG-GWO...")

for run in range(n_runs):
    print(f"--- Run {run + 1}/{n_runs} ---")
    
    # Initialize algorithms
    model_gwo = GWO.OriginalGWO(epoch=epoch, pop_size=pop_size)
    model_cggwo = CG_GWO(epoch=epoch, pop_size=pop_size)
    
    # Run GWO
    best_gwo = model_gwo.solve(problem_instance)
    results_gwo.append({
        'best_fitness': best_gwo.target.fitness,
        'solution': best_gwo.solution,
        'history': model_gwo.history.list_global_best_fit
    })
    
    # Run CG-GWO
    best_cggwo = model_cggwo.solve(problem_instance)
    results_cggwo.append({
        'best_fitness': best_cggwo.target.fitness,
        'solution': best_cggwo.solution,
        'history': model_cggwo.history.list_global_best_fit
    })

print("Optimization complete.")


### 6. Interpretability & Visualization

We plot the convergence curves (Best Fitness vs. Iterations) for both methods.
Then, we generate Confusion Matrices, ROC Curves, and Precision-Recall (PR) Curves for the best models found across all runs.
Finally, we visualize feature importance by showing which features were selected most frequently across runs.


In [ ]:
# Extract Best Models and Histories
best_run_gwo = min(results_gwo, key=lambda x: x['best_fitness'])
best_run_cggwo = min(results_cggwo, key=lambda x: x['best_fitness'])

# Plot Convergence Curves
plt.figure(figsize=(10, 6))

# Average Convergence Curves over 30 runs
avg_history_gwo = np.mean([r['history'] for r in results_gwo], axis=0)
avg_history_cggwo = np.mean([r['history'] for r in results_cggwo], axis=0)

plt.plot(avg_history_gwo, label="Original GWO", color="blue", linewidth=2)
plt.plot(avg_history_cggwo, label="CG-GWO", color="red", linewidth=2, linestyle="--")

plt.title("Average Convergence Curves (30 Runs)")
plt.xlabel("Iteration")
plt.ylabel("Fitness Value (Minimization)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, precision_recall_curve, auc

def evaluate_best_model(solution, model_name):
    binary_features, C, gamma = problem_instance.decode_solution(solution)
    selected_indices = np.where(binary_features == 1)[0]
    
    # Train final SVM
    svm = SVC(C=C, gamma=gamma, probability=True, random_state=42)
    svm.fit(X_train_scaled[:, selected_indices], y_train)
    
    # Predict
    y_pred = svm.predict(X_test_scaled[:, selected_indices])
    y_prob = svm.predict_proba(X_test_scaled[:, selected_indices])[:, 1]
    
    # Output Basic Metrics
    acc = accuracy_score(y_test, y_pred)
    if model_name in ["GWO", "CG-GWO"]: print(f"--- {model_name} Best Model Performance ---")
    if model_name in ["GWO", "CG-GWO"]: print(f"Accuracy: {acc:.4f}")
    if model_name in ["GWO", "CG-GWO"]: print(f"Selected Features ({len(selected_indices)}): {[X.columns[i] for i in selected_indices]}")
    if model_name in ["GWO", "CG-GWO"]: print(f"C: {C:.4f}, gamma: {gamma:.4f}\n")
    
    return y_pred, y_prob, acc, selected_indices, C, gamma

# Evaluate Best Models
y_pred_gwo, y_prob_gwo, acc_gwo, sel_idx_gwo, C_gwo, gamma_gwo = evaluate_best_model(best_run_gwo['solution'], "GWO")
y_pred_cggwo, y_prob_cggwo, acc_cggwo, sel_idx_cggwo, C_cggwo, gamma_cggwo = evaluate_best_model(best_run_cggwo['solution'], "CG-GWO")


In [ ]:
# Visualizations: Confusion Matrices
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

cm_gwo = confusion_matrix(y_test, y_pred_gwo)
disp_gwo = ConfusionMatrixDisplay(confusion_matrix=cm_gwo)
disp_gwo.plot(ax=ax[0], cmap='Blues')
ax[0].set_title('GWO Confusion Matrix')

cm_cggwo = confusion_matrix(y_test, y_pred_cggwo)
disp_cggwo = ConfusionMatrixDisplay(confusion_matrix=cm_cggwo)
disp_cggwo.plot(ax=ax[1], cmap='Oranges')
ax[1].set_title('CG-GWO Confusion Matrix')

plt.tight_layout()
plt.show()


In [ ]:
# Visualizations: ROC Curves
fpr_gwo, tpr_gwo, _ = roc_curve(y_test, y_prob_gwo)
roc_auc_gwo = auc(fpr_gwo, tpr_gwo)

fpr_cggwo, tpr_cggwo, _ = roc_curve(y_test, y_prob_cggwo)
roc_auc_cggwo = auc(fpr_cggwo, tpr_cggwo)

plt.figure(figsize=(8, 6))
plt.plot(fpr_gwo, tpr_gwo, color='blue', lw=2, label=f'GWO (AUC = {roc_auc_gwo:.4f})')
plt.plot(fpr_cggwo, tpr_cggwo, color='red', lw=2, linestyle='--', label=f'CG-GWO (AUC = {roc_auc_cggwo:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle=':')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


In [ ]:
# Visualizations: PR Curves
precision_gwo, recall_gwo, _ = precision_recall_curve(y_test, y_prob_gwo)
pr_auc_gwo = auc(recall_gwo, precision_gwo)

precision_cggwo, recall_cggwo, _ = precision_recall_curve(y_test, y_prob_cggwo)
pr_auc_cggwo = auc(recall_cggwo, precision_cggwo)

plt.figure(figsize=(8, 6))
plt.plot(recall_gwo, precision_gwo, color='blue', lw=2, label=f'GWO (PR AUC = {pr_auc_gwo:.4f})')
plt.plot(recall_cggwo, precision_cggwo, color='red', lw=2, linestyle='--', label=f'CG-GWO (PR AUC = {pr_auc_cggwo:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall (PR) Curve')
plt.legend(loc="lower left")
plt.grid(True)
plt.show()


In [ ]:
# Feature Importance (Selection Frequency across 30 runs)
def get_selection_frequencies(results_list):
    freqs = np.zeros(problem_instance.n_features)
    for r in results_list:
        binary_features, _, _ = problem_instance.decode_solution(r['solution'])
        freqs += binary_features
    return freqs

freqs_gwo = get_selection_frequencies(results_gwo)
freqs_cggwo = get_selection_frequencies(results_cggwo)

# Plot
features = X.columns
x_indices = np.arange(len(features))
width = 0.35

fig, ax = plt.subplots(figsize=(15, 8))
rects1 = ax.bar(x_indices - width/2, freqs_gwo, width, label='GWO', color='blue')
rects2 = ax.bar(x_indices + width/2, freqs_cggwo, width, label='CG-GWO', color='red')

ax.set_ylabel('Selection Frequency (out of 30 runs)')
ax.set_title('Feature Selection Frequency Comparison')
ax.set_xticks(x_indices)
ax.set_xticklabels(features, rotation=90)
ax.legend()

plt.tight_layout()
plt.show()


### 7. Statistical Testing

To prove statistical validity, we perform a **Wilcoxon Signed-Rank Test** on the accuracy results of the 30 independent runs between standard GWO and CG-GWO. This will tell us if CG-GWO's improvement is statistically significant (p-value < 0.05).


In [ ]:
from scipy.stats import wilcoxon

# Calculate accuracies for all 30 runs
acc_list_gwo = []
acc_list_cggwo = []

for r in results_gwo:
    _, _, acc, _, _, _ = evaluate_best_model(r['solution'], "GWO Run")
    acc_list_gwo.append(acc)
    
for r in results_cggwo:
    _, _, acc, _, _, _ = evaluate_best_model(r['solution'], "CG-GWO Run")
    acc_list_cggwo.append(acc)

# Perform Wilcoxon Signed-Rank Test
stat, p_value = wilcoxon(acc_list_gwo, acc_list_cggwo)

print("\n" + "="*50)
print(f"Wilcoxon Signed-Rank Test Results:")
print(f"Test Statistic: {stat:.4f}")
print(f"p-value: {p_value:.4e}")

if p_value < 0.05:
    print("Conclusion: There is a statistically significant difference between GWO and CG-GWO.")
else:
    print("Conclusion: There is NO statistically significant difference between GWO and CG-GWO.")
print("="*50)


### 8. Final Result Comparison

Summary table comparing the best configurations found by both algorithms.


In [ ]:
summary_df = pd.DataFrame({
    "Algorithm": ["Original GWO", "CG-GWO"],
    "Best Accuracy": [acc_gwo, acc_cggwo],
    "Best C": [C_gwo, C_cggwo],
    "Best Gamma": [gamma_gwo, gamma_cggwo],
    "Number of Selected Features": [len(sel_idx_gwo), len(sel_idx_cggwo)],
    "Selected Features": [
        ", ".join([X.columns[i] for i in sel_idx_gwo]),
        ", ".join([X.columns[i] for i in sel_idx_cggwo])
    ]
})

# Display table
summary_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
